# Telco Customer Churn — Evaluation

Loads the 5 tuned models saved by `03_Modeling.ipynb`, sweeps prediction thresholds instead of using the default 0.5 cutoff, and picks a final model + threshold using a business-driven rule: **Recall for the churn class (1) must be at least 0.80** — missing a real churner is treated as more costly than a false alarm — and among the thresholds that clear that bar, the one with the best F1 wins.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

### Load models and test data

Decision Tree and Random Forest were trained on **unscaled** features and must be evaluated on `x_test_raw.csv`. SVC, KNN and Logistic Regression were trained on **scaled** features (via `StandardScaler`) and must be evaluated on `x_test_scaled.csv`. Mixing these up — which the previous version of this notebook did, by loading a single `x_test.csv` for every model — silently breaks the tree models' predictions (see the note in `03_Modeling.ipynb`).

In [4]:
import joblib

best_tree_model = joblib.load("../models/decision_tree.pkl")
best_rf_model = joblib.load("../models/random_forest.pkl")
best_svc_model = joblib.load("../models/svc.pkl")
best_KNN_model = joblib.load("../models/knn.pkl")
best_lg_model = joblib.load("../models/logistic_regression.pkl")

x_test_raw = pd.read_csv("../data/x_test_raw.csv").values
x_test_scaled = pd.read_csv("../data/x_test_scaled.csv").values
y_test = pd.read_csv("../data/y_test.csv").values.ravel()

### Models and which test set each one needs

Pairing each model with its correct `X_test` here, once, so every loop below stays correct automatically instead of relying on remembering which model needs which scale.

In [5]:
models = {
    "Decision Tree": (best_tree_model, x_test_raw),
    "Random Forest": (best_rf_model, x_test_raw),
    "SVC": (best_svc_model, x_test_scaled),
    "KNN": (best_KNN_model, x_test_scaled),
    "Logistic Regression": (best_lg_model, x_test_scaled),
}

### Threshold sweep

For each model, sweep thresholds from 0.20 to 0.70 and compute Accuracy, Precision, Recall and F1 for the churn class at each one, using that model's own correct `X_test`.

In [6]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

threshold_results = []

for model_name, (model, x_te) in models.items():

    y_prob = model.predict_proba(x_te)[:, 1]

    for threshold in thresholds:

        y_pred = (y_prob >= threshold).astype(int)

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision_1": precision_score(y_test, y_pred, zero_division=0),
            "Recall_1": recall_score(y_test, y_pred, zero_division=0),
            "F1_1": f1_score(y_test, y_pred, zero_division=0)
        })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

,Model,Threshold,Accuracy,Precision_1,Recall_1,F1_1
0,Decision Tree,0.20,0.571327,0.378178,0.954545,0.541730
1,Decision Tree,0.25,0.655075,0.429114,0.906417,0.582474
2,Decision Tree,0.30,0.655075,0.429114,0.906417,0.582474
3,Decision Tree,0.35,0.655075,0.429114,0.906417,0.582474
4,Decision Tree,0.40,0.664301,0.435798,0.898396,0.586900
5,Decision Tree,0.45,0.696948,0.461650,0.852941,0.599061
6,Decision Tree,0.50,0.732434,0.497453,0.783422,0.608515
7,Decision Tree,0.55,0.765791,0.545267,0.708556,0.616279
8,Decision Tree,0.60,0.769340,0.551797,0.697861,0.616293
9,Decision Tree,0.65,0.787083,0.593434,0.628342,0.610390


### Final comparison table — best threshold per model

One row per model: the threshold with the best F1 among those meeting the Recall ≥ 0.80 business requirement. This is the table for the README / portfolio write-up — it shows each model's *own* best achievable trade-off, not just the single overall winner.

In [7]:
MIN_RECALL = 0.80

best_per_model = (
    threshold_df[threshold_df["Recall_1"] >= MIN_RECALL]
    .sort_values(["F1_1", "Threshold"], ascending=[False, True])
    .groupby("Model", as_index=False)
    .first()
    .sort_values("F1_1", ascending=False)
    .reset_index(drop=True)
)
best_per_model

,Model,Threshold,Accuracy,Precision_1,Recall_1,F1_1
0,Logistic Regression,0.50,0.755145,0.525130,0.810160,0.637224
1,Random Forest,0.50,0.718950,0.482704,0.820856,0.607921
2,Decision Tree,0.45,0.696948,0.461650,0.852941,0.599061
3,SVC,0.25,0.706884,0.470138,0.820856,0.597858
4,KNN,0.30,0.694109,0.456884,0.807487,0.583575


### Overall best model

The top row of `best_per_model` above is the single best model+threshold combination under the Recall ≥ 0.80 constraint.

In [8]:
best_model_name = best_per_model.iloc[0]["Model"]
best_model, best_x_test = models[best_model_name]
best_threshold = best_per_model.iloc[0]["Threshold"]

print(f"Best model: {best_model_name} @ threshold {best_threshold}")

Best model: Logistic Regression @ threshold 0.5


In [9]:
y_prob = best_model.predict_proba(best_x_test)[:, 1]
y_pred_final = (y_prob >= best_threshold).astype(int)

In [10]:
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1035
           1       0.53      0.81      0.64       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409



## Conclusion

**Chosen model:** the model and threshold selected above, under the constraint of catching at least 80% of actual churners (Recall ≥ 0.80).

**Business trade-off:** at that Recall level, Precision for the churn class is well below 1.0 — meaning a meaningful share of customers flagged as "will churn" will not actually churn. This is an accepted trade-off: in a retention-campaign setting, the cost of a false alarm (an unnecessary retention offer) is assumed to be much lower than the cost of missing a real churner (lost revenue, no chance to intervene). If that assumption doesn't hold for a given business context, the `MIN_RECALL` constraint above should be revisited rather than the model itself.

**Note:** re-run this notebook after re-running the fixed `03_Modeling.ipynb` — Decision Tree and Random Forest were being evaluated on incorrectly-scaled test data in the original version, so their numbers here will change once the scaling bug fix (see `03_Modeling.ipynb`) is applied and both notebooks are re-executed top to bottom.